In [1]:
# Implementation of a KAN for Brain Voxel Classification (1-vs-Rest)
# This notebook implements a Kolmogorov-Arnold Network (KAN) for 
# brain voxel classification using a 1-vs-rest approach.

# Cell 1: Initialize the environment and import libraries
import torch
from kan import *
import numpy as np
import matplotlib.pyplot as plt
import os
import time
from sklearn.metrics import precision_recall_curve, average_precision_score, f1_score
from sklearn.preprocessing import StandardScaler
from datetime import datetime
import pandas as pd
import random
import glob
from tqdm import tqdm

# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU")

# Create folders for saving results
os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('video_img', exist_ok=True)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

Using GPU: NVIDIA RTX A6000


In [2]:
# Cell 2: Hyperparameters Configuration

class Config:
    def __init__(self):
        # Data parameters
        self.feature_dim = 341  # Number of input features
        self.num_classes = 2    # Binary classification (1-vs-rest)
        self.negative_ratio = 10  # Ratio of negative to positive samples (1:k)
        self.val_negative_ratio = 5  # Validation set negative to positive ratio
        self.test_size = 0.2    # Fraction of data to use for testing
        self.random_state = 42  # Random seed for data splitting
        self.normalize_features = True  # Whether to standardize features
        
        # KAN model parameters
        self.network_width = [self.feature_dim, 256, 128, 64, self.num_classes]  # Network architecture
        self.grid_size = 20     # Fixed grid size (no grid expansion to avoid oscillation)
        self.k_value = 3        # Number of basis functions per dimension
        
        # Training parameters
        self.optimizer = "Adam"  # Optimizer type (Adam, SGD, etc.)
        self.learning_rate = 0.005  # Learning rate for optimizer
        self.weight_decay = 0.0001  # L2 regularization coefficient
        self.lambda_reg = 0.01    # Weight regularization coefficient
        self.lambda_entropy = 5.0  # Entropy regularization coefficient
        self.batch_size = 64      # Batch size for training
        self.train_steps = 100    # Number of training steps
        self.early_stopping = True  # Whether to use early stopping
        self.patience = 10        # Early stopping patience
        self.min_delta = 0.001    # Minimum improvement for early stopping

        # Class weighting for imbalanced data
        self.class_weights = torch.tensor([1.0, self.negative_ratio * 0.5], dtype=torch.float32)  # [pos_weight, neg_weight]
        
        # Pruning and fine-tuning
        self.enable_pruning = True  # Whether to prune the model after training
        self.fine_tune_steps = 50   # Number of fine-tuning steps after pruning
        
        # Evaluation metrics
        self.prediction_threshold = 0.5  # Threshold for binary predictions
        
        # Visualization parameters
        self.save_figures = True  # Whether to save figures
        self.img_folder = 'video_img'  # Folder to save training visualization images
        self.plot_frequency = 10   # How often to plot the model during training
        self.video_fps = 10        # Frames per second for training video
        
        # Symbolic regression parameters
        self.enable_symbolic = True  # Whether to extract symbolic expressions
        self.symbolic_library = ['x', 'x^2', 'exp', 'log', 'sqrt', 'sin', 'tanh', 'abs']  # Function library

    def display(self):
        """Display the current configuration"""
        print("=== KAN Brain Classification Configuration ===")
        print(f"Network Structure: {self.network_width}")
        print(f"Grid Size: {self.grid_size}, K Value: {self.k_value}")
        print(f"Negative to Positive Ratio: {self.negative_ratio}:1")
        print(f"Training Steps: {self.train_steps}, Batch Size: {self.batch_size}")
        print(f"Regularization: λ_reg={self.lambda_reg}, λ_entropy={self.lambda_entropy}")
        print(f"Learning Rate: {self.learning_rate}, Weight Decay: {self.weight_decay}")
        print(f"Early Stopping: {self.early_stopping} (patience={self.patience}, min_delta={self.min_delta})")
        print(f"Class Weights: {self.class_weights}")
        print("===============================================")

# Create a configuration object
config = Config()
config.display()

=== KAN Brain Classification Configuration ===
Network Structure: [341, 256, 128, 64, 2]
Grid Size: 20, K Value: 3
Negative to Positive Ratio: 10:1
Training Steps: 100, Batch Size: 64
Regularization: λ_reg=0.01, λ_entropy=5.0
Learning Rate: 0.005, Weight Decay: 0.0001
Early Stopping: True (patience=10, min_delta=0.001)
Class Weights: tensor([1., 5.])


In [3]:
# Cell 3: Data Loading and Processing Functions

def load_label_index(index_file):
    """Load label index file containing voxel counts for each label"""
    label_info = {}
    with open(index_file, 'r') as f:
        # Skip header
        next(f)
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 3:
                label_id = int(parts[0])
                voxel_count = int(parts[1])
                filename = parts[2] if parts[2] else None
                label_info[label_id] = {'count': voxel_count, 'filename': filename}
    return label_info

def get_label_file_path(label_id, is_validation=False, base_dir=None):
    """Get file path for a specific label"""
    if base_dir is None:
        if is_validation:
            base_dir = 'path_to_val_data'  # Replace with your validation data path
        else:
            base_dir = 'path_to_train_data'  # Replace with your training data path
    
    pattern = os.path.join(base_dir, f"label_{label_id}_count_*_voxels.npy")
    matches = glob.glob(pattern)
    return matches[0] if matches else None

def create_1vs_rest_dataset(target_label_id, config, verbose=True):
    """
    Create a 1-vs-rest dataset for binary classification of a specific label
    
    Args:
        target_label_id: ID of the target label (positive class)
        config: Configuration object with hyperparameters
        verbose: Whether to print progress information
    
    Returns:
        dataset: Dictionary containing training and testing data
    """
    if verbose:
        print(f"Creating 1-vs-rest dataset for label {target_label_id}...")
    
    # 1. Load the target label data (positive samples)
    target_file = get_label_file_path(target_label_id)
    if not target_file:
        raise ValueError(f"Label {target_label_id} has no corresponding data file")
    
    positive_samples = np.load(target_file)
    num_positive = len(positive_samples)
    
    if verbose:
        print(f"  Loaded {num_positive} positive samples from {target_file}")
    
    # 2. Calculate how many negative samples we need based on the ratio
    num_negative = num_positive * config.negative_ratio
    
    # 3. Load negative samples from other labels
    # In a real implementation, you would load actual label data from other labels
    # For now, we'll simulate by creating synthetic negative samples
    
    # Get a list of all label IDs except the target
    all_label_ids = list(range(1, 102))  # Assuming labels are 1-101
    other_labels = [l for l in all_label_ids if l != target_label_id]
    
    # Randomly select which labels to use as negatives
    random.shuffle(other_labels)
    selected_labels = other_labels[:min(10, len(other_labels))]  # Use up to 10 other labels
    
    negative_samples = []
    negative_count = 0
    
    # For each selected label, load its data and add to negative samples
    for other_label_id in selected_labels:
        other_file = get_label_file_path(other_label_id)
        if not other_file:
            continue
            
        try:
            # Load data for this label
            label_samples = np.load(other_file)
            
            # Calculate samples to take from this label
            samples_needed = min(len(label_samples), int(num_negative / len(selected_labels)))
            
            # Randomly sample if we have more than needed
            if samples_needed < len(label_samples):
                indices = np.random.choice(len(label_samples), samples_needed, replace=False)
                sampled = label_samples[indices]
            else:
                sampled = label_samples
                
            negative_samples.append(sampled)
            negative_count += len(sampled)
            
            if verbose:
                print(f"  Added {len(sampled)} negative samples from label {other_label_id}")
                
            # Check if we have enough negative samples
            if negative_count >= num_negative:
                break
                
        except Exception as e:
            print(f"  Error loading data for label {other_label_id}: {str(e)}")
    
    # Combine all negative samples
    if negative_samples:
        negative_samples = np.vstack(negative_samples)
        
        # If we have more than needed, randomly sample
        if len(negative_samples) > num_negative:
            indices = np.random.choice(len(negative_samples), int(num_negative), replace=False)
            negative_samples = negative_samples[indices]
    else:
        # If no negative samples were loaded, create synthetic ones
        if verbose:
            print("  Warning: No negative samples loaded. Creating synthetic negative samples.")
        negative_samples = np.random.randn(int(num_negative), config.feature_dim)
    
    if verbose:
        print(f"  Final negative sample count: {len(negative_samples)}")
    
    # 4. Create feature data X and labels y
    X = np.vstack([positive_samples, negative_samples])
    
    # Create labels: 1 for positive class, 0 for negative class
    y_positive = np.ones((num_positive, 1))
    y_negative = np.zeros((len(negative_samples), 1))
    y = np.vstack([y_positive, y_negative])
    
    # Convert to integer class labels for CrossEntropyLoss
    y_class = y.flatten().astype(np.int64)
    
    # 5. Shuffle the data
    indices = np.arange(X.shape[0])
    np.random.shuffle(indices)
    X = X[indices]
    y_class = y_class[indices]
    
    # 6. Normalize features if requested
    if config.normalize_features:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
    
    # 7. Split into training and testing sets
    test_size = int(X.shape[0] * config.test_size)
    X_train, X_test = X[:-test_size], X[-test_size:]
    y_train, y_test = y_class[:-test_size], y_class[-test_size:]
    
    if verbose:
        print(f"  Total dataset size: {X.shape[0]} samples")
        print(f"  Training set: {X_train.shape[0]} samples")
        print(f"  Testing set: {X_test.shape[0]} samples")
        print(f"  Feature dimension: {X.shape[1]}")
        
        # Calculate class distribution
        train_pos = np.sum(y_train == 1)
        train_neg = np.sum(y_train == 0)
        test_pos = np.sum(y_test == 1)
        test_neg = np.sum(y_test == 0)
        
        print(f"  Training set class distribution: {train_pos} positive, {train_neg} negative")
        print(f"  Testing set class distribution: {test_pos} positive, {test_neg} negative")
    
    # 8. Convert to PyTorch tensors and move to device
    train_inputs = torch.tensor(X_train, dtype=torch.float32).to(device)
    train_labels = torch.tensor(y_train, dtype=torch.long).to(device)
    test_inputs = torch.tensor(X_test, dtype=torch.float32).to(device)
    test_labels = torch.tensor(y_test, dtype=torch.long).to(device)
    
    # 9. Create dataset dictionary
    dataset = {
        'train_input': train_inputs,
        'train_label': train_labels,
        'test_input': test_inputs,
        'test_label': test_labels,
        'label_id': target_label_id
    }
    
    return dataset

# Simulate data loading for demonstration
# In a real implementation, replace this with actual data loading
def create_demo_dataset(target_label_id, config, verbose=True):
    """Create a simulated dataset for demonstration purposes"""
    if verbose:
        print(f"Creating simulated dataset for label {target_label_id}...")
    
    # Set dimensions
    num_positive = 500  # Simulate 500 positive samples
    num_negative = num_positive * config.negative_ratio
    
    # Create synthetic features (random for demonstration)
    positive_features = np.random.randn(num_positive, config.feature_dim)
    # Make negative samples slightly different distribution
    negative_features = np.random.randn(int(num_negative), config.feature_dim) * 1.2 + 0.5
    
    # Combine features and create labels
    X = np.vstack([positive_features, negative_features])
    y_positive = np.ones(num_positive, dtype=np.int64)
    y_negative = np.zeros(int(num_negative), dtype=np.int64)
    y = np.concatenate([y_positive, y_negative])
    
    # Shuffle the data
    indices = np.arange(X.shape[0])
    np.random.shuffle(indices)
    X = X[indices]
    y = y[indices]
    
    # Split into training and testing sets
    test_size = int(X.shape[0] * config.test_size)
    X_train, X_test = X[:-test_size], X[-test_size:]
    y_train, y_test = y[:-test_size], y[-test_size:]
    
    if verbose:
        print(f"  Created simulated dataset with {X.shape[0]} samples")
        print(f"  Training set: {X_train.shape[0]} samples")
        print(f"  Testing set: {X_test.shape[0]} samples")
    
    # Convert to PyTorch tensors and move to device
    train_inputs = torch.tensor(X_train, dtype=torch.float32).to(device)
    train_labels = torch.tensor(y_train, dtype=torch.long).to(device)
    test_inputs = torch.tensor(X_test, dtype=torch.float32).to(device)
    test_labels = torch.tensor(y_test, dtype=torch.long).to(device)
    
    # Create dataset dictionary
    dataset = {
        'train_input': train_inputs,
        'train_label': train_labels,
        'test_input': test_inputs,
        'test_label': test_labels,
        'label_id': target_label_id
    }
    
    return dataset

# Load the dataset for a specific label
target_label = 15  # Change this to the label you want to classify
dataset = create_demo_dataset(target_label, config)

# Print dataset summary
print(f"Train data shape: {dataset['train_input'].shape}")
print(f"Train label shape: {dataset['train_label'].shape}")
print(f"Test data shape: {dataset['test_input'].shape}")
print(f"Test label shape: {dataset['test_label'].shape}")
print("====================================")

Creating simulated dataset for label 15...
  Created simulated dataset with 5500 samples
  Training set: 4400 samples
  Testing set: 1100 samples
Train data shape: torch.Size([4400, 341])
Train label shape: torch.Size([4400])
Test data shape: torch.Size([1100, 341])
Test label shape: torch.Size([1100])


In [4]:
# Cell 4 (Simplified): KAN Model Initialization

import torch
from kan import *
import matplotlib.pyplot as plt

# 使用与示例相同的简单初始化方式
model = KAN(width=[341, 256, 128, 2], grid=10, k=3, seed=0, device=device)

# 执行前向传播来初始化模型
_ = model(dataset['train_input'][:10])

print(f"KAN模型已创建: 输入维度={341}, 输出维度=2")
print(f"网格大小: 10, k值: 3")
print(f"总参数数量: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

# 尝试绘制模型结构
# try:
#     model.plot(beta=100, scale=1, in_vars=['F1', 'F2', 'F3'], out_vars=['Neg', 'Pos'])
# except:
#     print("无法绘制模型结构图，但这不影响训练")

checkpoint directory created: ./model
saving model version 0.0
KAN模型已创建: 输入维度=341, 输出维度=2
网格大小: 10, k值: 3
总参数数量: 2286080


In [5]:
# Cell 5: KAN Training Functions and Metrics

# Define evaluation metrics for binary classification
def accuracy_metric(model, data_input, data_label):
    """Calculate classification accuracy"""
    with torch.no_grad():
        outputs = model(data_input)
        predictions = torch.argmax(outputs, dim=1)
        return torch.mean((predictions == data_label).float())

def precision_metric(model, data_input, data_label):
    """Calculate precision (TP / (TP + FP))"""
    with torch.no_grad():
        outputs = model(data_input)
        predictions = torch.argmax(outputs, dim=1)
        true_positives = torch.sum((predictions == 1) & (data_label == 1)).float()
        false_positives = torch.sum((predictions == 1) & (data_label == 0)).float()
        if true_positives + false_positives == 0:
            return torch.tensor(0.0, device=device)
        return true_positives / (true_positives + false_positives)

def recall_metric(model, data_input, data_label):
    """Calculate recall (TP / (TP + FN))"""
    with torch.no_grad():
        outputs = model(data_input)
        predictions = torch.argmax(outputs, dim=1)
        true_positives = torch.sum((predictions == 1) & (data_label == 1)).float()
        false_negatives = torch.sum((predictions == 0) & (data_label == 1)).float()
        if true_positives + false_negatives == 0:
            return torch.tensor(0.0, device=device)
        return true_positives / (true_positives + false_negatives)

def f1_metric(model, data_input, data_label):
    """Calculate F1 score (harmonic mean of precision and recall)"""
    prec = precision_metric(model, data_input, data_label)
    rec = recall_metric(model, data_input, data_label)
    if prec + rec == 0:
        return torch.tensor(0.0, device=device)
    return 2 * prec * rec / (prec + rec)

def auc_pr_metric(model, data_input, data_label):
    """Calculate Area Under the Precision-Recall Curve (AUC-PR)"""
    with torch.no_grad():
        outputs = model(data_input)
        probabilities = torch.softmax(outputs, dim=1)[:, 1]  # Probability of positive class
        labels = data_label.cpu().numpy()
        probs = probabilities.cpu().numpy()
        
        # Calculate AUC-PR using scikit-learn
        if np.sum(labels) == 0:  # No positive samples
            return torch.tensor(0.0, device=device)
        
        auc_score = average_precision_score(labels, probs)
        return torch.tensor(auc_score, device=device)

def train_acc_fn():
    """Wrapper for training accuracy"""
    return accuracy_metric(model, dataset['train_input'], dataset['train_label'])

def train_f1_fn():
    """Wrapper for training F1 score"""
    return f1_metric(model, dataset['train_input'], dataset['train_label'])

def test_acc_fn():
    """Wrapper for test accuracy"""
    return accuracy_metric(model, dataset['test_input'], dataset['test_label'])

def test_f1_fn():
    """Wrapper for test F1 score"""
    return f1_metric(model, dataset['test_input'], dataset['test_label'])

def test_auc_pr_fn():
    """Wrapper for test AUC-PR"""
    return auc_pr_metric(model, dataset['test_input'], dataset['test_label'])

def train_with_early_stopping(model, dataset, config):
    """
    Train the KAN model with early stopping
    
    Args:
        model: KAN model instance
        dataset: Dictionary containing training and testing data
        config: Configuration object with hyperparameters
    
    Returns:
        results: Dictionary containing training results
    """
    print("Starting training with early stopping...")
    
    # Define metrics to track
    metrics = (train_acc_fn, train_f1_fn, test_acc_fn, test_f1_fn, test_auc_pr_fn)
    metric_names = ['train_acc', 'train_f1', 'test_acc', 'test_f1', 'test_auc_pr']
    
    # Set up weight balancing for CrossEntropyLoss
    # Move weights to the same device as the model
    class_weights = config.class_weights.to(device)
    
    # Create loss function with weights
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
    
    # Early stopping variables
    best_score = 0.0
    best_model = None
    patience_counter = 0
    
    # Setup for KAN's fit method
    train_options = {
        'opt': config.optimizer,
        'metrics': metrics,
        'metric_names': metric_names,
        'loss_fn': criterion,
        'steps': config.train_steps,
        'lamb': config.lambda_reg,
        'lamb_entropy': config.lambda_entropy,
        'batch_size': config.batch_size,
        'lr': config.learning_rate,
        'weight_decay': config.weight_decay,
        'save_fig': config.save_figures,
        'img_folder': config.img_folder
    }
    
    # If early stopping is disabled, train for the full number of steps
    if not config.early_stopping:
        results = model.fit(dataset, **train_options)
        return results, model
    
    # Create a custom training loop with early stopping
    # First, initialize optimizer and schedulers
    if config.optimizer == 'Adam':
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=config.learning_rate,
            weight_decay=config.weight_decay
        )
    else:
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=config.learning_rate,
            weight_decay=config.weight_decay
        )
    
    # Initialize results dictionary
    results = {name: [] for name in metric_names}
    results['loss'] = []
    
    # Get data
    train_input = dataset['train_input']
    train_label = dataset['train_label']
    
    # Training loop with early stopping
    step = 0
    stop_training = False
    
    while step < config.train_steps and not stop_training:
        # Forward pass and loss calculation
        model.train()
        
        # Sample a batch
        if train_input.shape[0] <= config.batch_size:
            batch_input = train_input
            batch_label = train_label
        else:
            indices = torch.randperm(train_input.shape[0])[:config.batch_size]
            batch_input = train_input[indices]
            batch_label = train_label[indices]
        
        optimizer.zero_grad()
        
        outputs = model(batch_input)
        loss = criterion(outputs, batch_label)
        
        # Add regularization terms
        if config.lambda_reg > 0:
            l1_reg = 0
            for p in model.parameters():
                l1_reg += torch.sum(torch.abs(p))
            loss += config.lambda_reg * l1_reg
        
        # 修复 - 处理不同版本KAN的熵正则化
        if config.lambda_entropy > 0:
            entropy_reg = 0
            try:
                # 如果是原版KAN结构，使用layers属性
                if hasattr(model, 'layers'):
                    for layer in model.layers:
                        if hasattr(layer, 'rbf'):
                            entropy_reg += layer.rbf.entropy()
                # 如果是MultKAN结构，尝试直接使用entropy()方法
                elif hasattr(model, 'entropy'):
                    entropy_reg = model.entropy()
                # 否则，使用可能的其他结构
                elif hasattr(model, 'KAN') and hasattr(model.KAN, 'layers'):
                    for layer in model.KAN.layers:
                        if hasattr(layer, 'rbf'):
                            entropy_reg += layer.rbf.entropy()
                # 如果找不到任何熵方法，输出警告并跳过熵正则化
                else:
                    print("警告: 无法找到计算熵的方法，跳过熵正则化")
                    config.lambda_entropy = 0
                    entropy_reg = 0
                    
                loss += config.lambda_entropy * entropy_reg
            except Exception as e:
                print(f"熵正则化计算出错: {e}")
                config.lambda_entropy = 0
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        # Calculate metrics every few steps or at the last step
        if (step + 1) % 5 == 0 or step == config.train_steps - 1:
            model.eval()
            
            # Calculate all metrics
            metric_values = [metric_fn().item() for metric_fn in metrics]
            
            # Store metrics
            for name, value in zip(metric_names, metric_values):
                results[name].append(value)
            
            results['loss'].append(loss.item())
            
            # Print progress
            progress_str = f"Step {step+1}/{config.train_steps}"
            for name, value in zip(metric_names, metric_values):
                progress_str += f", {name}: {value:.4f}"
            print(progress_str)
            
            # Check for early stopping (using F1 score on validation set)
            current_score = metric_values[3]  # test_f1 index
            
            if current_score > best_score + config.min_delta:
                best_score = current_score
                patience_counter = 0
                # Save the best model
                best_model = copy.deepcopy(model)
                print(f"New best model with test F1: {best_score:.4f}")
            else:
                patience_counter += 1
                if patience_counter >= config.patience:
                    print(f"Early stopping at step {step+1} with best test F1: {best_score:.4f}")
                    stop_training = True
        
        # Save model visualization if requested
        if config.save_figures and (step + 1) % config.plot_frequency == 0:
            model.eval()
            fig = model.plot(
                beta=50,
                scale=1,
                in_vars=feature_names[:min(10, config.feature_dim)],
                out_vars=output_names
            )
            plt.savefig(f'{config.img_folder}/{step+1}.jpg', dpi=150, bbox_inches='tight')
            plt.close(fig)
        
        step += 1
    
    # Return the best model if early stopping was used
    if best_model is not None:
        if stop_training:
            print("Restoring best model from early stopping")
            model.load_state_dict(best_model.state_dict())
    
    return results, model

# Print available metrics
print("Available metrics for training:")
print("  - Accuracy: Percentage of correctly classified samples")
print("  - Precision: Proportion of positive identifications that were actually correct")
print("  - Recall: Proportion of actual positives that were correctly identified")
print("  - F1 Score: Harmonic mean of precision and recall")
print("  - AUC-PR: Area Under the Precision-Recall Curve")
print("====================================")

Available metrics for training:
  - Accuracy: Percentage of correctly classified samples
  - Precision: Proportion of positive identifications that were actually correct
  - Recall: Proportion of actual positives that were correctly identified
  - F1 Score: Harmonic mean of precision and recall
  - AUC-PR: Area Under the Precision-Recall Curve


In [6]:
# Cell 6: KAN Model Training

print(f"Starting KAN training for label {target_label} (1-vs-rest classification)")
print("--------------------------------------------------------------------")

# Record start time
start_time = time.time()

# Train the model with early stopping
results, model = train_with_early_stopping(model, dataset, config)

# Calculate training time
training_time = time.time() - start_time
print(f"Training completed in {training_time:.2f} seconds")

# Save the final model
torch.save(model.state_dict(), f'models/kan_label{target_label}_final.pt')
print(f"Model saved to 'models/kan_label{target_label}_final.pt'")

# Final evaluation
model.eval()
with torch.no_grad():
    train_outputs = model(dataset['train_input'])
    test_outputs = model(dataset['test_input'])
    
    # Get predictions (class with highest probability)
    train_preds = torch.argmax(train_outputs, dim=1)
    test_preds = torch.argmax(test_outputs, dim=1)
    
    # Calculate final metrics
    train_acc = torch.mean((train_preds == dataset['train_label']).float()).item()
    test_acc = torch.mean((test_preds == dataset['test_label']).float()).item()
    
    # Calculate F1 scores
    train_f1 = f1_metric(model, dataset['train_input'], dataset['train_label']).item()
    test_f1 = f1_metric(model, dataset['test_input'], dataset['test_label']).item()
    
    # Calculate AUC-PR
    test_auc_pr = auc_pr_metric(model, dataset['test_input'], dataset['test_label']).item()

print("\nFinal evaluation results:")
print(f"  Training accuracy: {train_acc:.4f}")
print(f"  Test accuracy: {test_acc:.4f}")
print(f"  Training F1 score: {train_f1:.4f}")
print(f"  Test F1 score: {test_f1:.4f}")
print(f"  Test AUC-PR: {test_auc_pr:.4f}")
print("--------------------------------------------------------------------")

Starting KAN training for label 15 (1-vs-rest classification)
--------------------------------------------------------------------
Starting training with early stopping...
警告: 无法找到计算熵的方法，跳过熵正则化
Step 5/100, train_acc: 0.9093, train_f1: 0.0000, test_acc: 0.9082, test_f1: 0.0000, test_auc_pr: 0.9945
Step 10/100, train_acc: 0.9093, train_f1: 0.0000, test_acc: 0.9082, test_f1: 0.0000, test_auc_pr: 0.7741


NameError: name 'feature_names' is not defined

In [ ]:
# Cell 7: Results Visualization

# Create a function to plot the training results
def plot_training_results(results):
    """Plot the training metrics and loss"""
    plt.figure(figsize=(15, 10))
    
    # Plot accuracy
    plt.subplot(2, 2, 1)
    plt.plot(results['train_acc'], label='Training Accuracy')
    plt.plot(results['test_acc'], label='Test Accuracy')
    plt.title('Accuracy vs. Training Steps')
    plt.xlabel('Steps (x5)')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    # Plot F1 score
    plt.subplot(2, 2, 2)
    plt.plot(results['train_f1'], label='Training F1')
    plt.plot(results['test_f1'], label='Test F1')
    plt.title('F1 Score vs. Training Steps')
    plt.xlabel('Steps (x5)')
    plt.ylabel('F1 Score')
    plt.legend()
    plt.grid(True)
    
    # Plot AUC-PR
    plt.subplot(2, 2, 3)
    plt.plot(results['test_auc_pr'], label='Test AUC-PR')
    plt.title('AUC-PR vs. Training Steps')
    plt.xlabel('Steps (x5)')
    plt.ylabel('AUC-PR')
    plt.legend()
    plt.grid(True)
    
    # Plot Loss
    plt.subplot(2, 2, 4)
    if 'loss' in results:
        plt.plot(results['loss'], label='Training Loss')
        plt.title('Loss vs. Training Steps')
        plt.xlabel('Steps (x5)')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)
    
    plt.tight_layout()
    return plt.gcf()

# Plot the results
fig = plot_training_results(results)
plt.savefig(f'results/training_metrics_label{target_label}.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot confusion matrix
def plot_confusion_matrix(model, dataset):
    """Plot confusion matrix for binary classification"""
    model.eval()
    with torch.no_grad():
        # Get predictions
        test_outputs = model(dataset['test_input'])
        test_preds = torch.argmax(test_outputs, dim=1).cpu().numpy()
        test_labels = dataset['test_label'].cpu().numpy()
        
        # Calculate confusion matrix
        cm = np.zeros((2, 2), dtype=int)
        for i in range(len(test_labels)):
            cm[test_labels[i], test_preds[i]] += 1
        
        # Plot
        plt.figure(figsize=(8, 6))
        plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
        plt.title(f'Confusion Matrix (Label {dataset["label_id"]})')
        plt.colorbar()
        
        classes = ['Negative', 'Positive']
        tick_marks = np.arange(len(classes))
        plt.xticks(tick_marks, classes, rotation=45)
        plt.yticks(tick_marks, classes)
        
        # Add text annotations
        thresh = cm.max() / 2.
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, format(cm[i, j], 'd'),
                        horizontalalignment="center",
                        color="white" if cm[i, j] > thresh else "black")
        
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        
        return plt.gcf(), cm

# Plot confusion matrix
conf_fig, cm = plot_confusion_matrix(model, dataset)
plt.savefig(f'results/confusion_matrix_label{target_label}.png', dpi=300, bbox_inches='tight')
plt.show()

# Calculate additional metrics from confusion matrix
tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print("\nConfusion Matrix Analysis:")
print(f"  True Positives: {tp}")
print(f"  False Positives: {fp}")
print(f"  True Negatives: {tn}")
print(f"  False Negatives: {fn}")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall/Sensitivity: {recall:.4f}")
print(f"  Specificity: {specificity:.4f}")
print(f"  F1 Score: {f1:.4f}")

# Plot Precision-Recall curve
def plot_pr_curve(model, dataset):
    """Plot precision-recall curve"""
    model.eval()
    with torch.no_grad():
        # Get predicted probabilities for the positive class
        test_outputs = model(dataset['test_input'])
        test_probs = torch.softmax(test_outputs, dim=1)[:, 1].cpu().numpy()
        test_labels = dataset['test_label'].cpu().numpy()
        
        # Calculate precision-recall curve
        precision, recall, thresholds = precision_recall_curve(test_labels, test_probs)
        
        # Calculate AUC-PR
        auc_pr = average_precision_score(test_labels, test_probs)
        
        # Find the threshold that gives the best F1 score
        f1_scores = 2 * precision * recall / (precision + recall + 1e-10)
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        best_f1 = f1_scores[best_idx]
        
        # Plot
        plt.figure(figsize=(8, 6))
        plt.plot(recall, precision, label=f'AUC-PR = {auc_pr:.4f}')
        plt.scatter(recall[best_idx], precision[best_idx], color='red', 
                   label=f'Best F1 = {best_f1:.4f} (threshold = {best_threshold:.4f})')
        
        # Add baseline (random classifier)
        pos_ratio = np.mean(test_labels)
        plt.plot([0, 1], [pos_ratio, pos_ratio], 'k--', label=f'Baseline (ratio = {pos_ratio:.4f})')
        
        plt.title(f'Precision-Recall Curve (Label {dataset["label_id"]})')
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.legend()
        plt.grid(True)
        
        return plt.gcf(), best_threshold, best_f1

# Plot Precision-Recall curve
pr_fig, best_threshold, best_f1 = plot_pr_curve(model, dataset)
plt.savefig(f'results/pr_curve_label{target_label}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPrecision-Recall Curve Analysis:")
print(f"  AUC-PR: {test_auc_pr:.4f}")
print(f"  Best F1 Score: {best_f1:.4f} (at threshold = {best_threshold:.4f})")
print(f"  Recommended threshold for optimal F1: {best_threshold:.4f}")

In [ ]:
# Cell 8: Model Pruning and Fine-tuning

if config.enable_pruning:
    print("\nPruning and fine-tuning the model...")
    
    # Save the original model for comparison
    original_model = copy.deepcopy(model)
    
    # Prune the model (remove unimportant weights)
    model = model.prune()
    
    print("Model pruned. Evaluating pruned model before fine-tuning...")
    
    # Evaluate pruned model before fine-tuning
    model.eval()
    with torch.no_grad():
        test_outputs = model(dataset['test_input'])
        test_preds = torch.argmax(test_outputs, dim=1)
        test_acc_pruned = torch.mean((test_preds == dataset['test_label']).float()).item()
        test_f1_pruned = f1_metric(model, dataset['test_input'], dataset['test_label']).item()
    
    print(f"  Pruned model test accuracy: {test_acc_pruned:.4f}")
    print(f"  Pruned model test F1 score: {test_f1_pruned:.4f}")
    
    # Fine-tune the pruned model
    print("\nFine-tuning the pruned model...")
    
    # Create loss function with weights
    class_weights = config.class_weights.to(device)
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
    
    # Run fine-tuning with the same loss function, but for fewer steps
    fine_tune_options = {
        'opt': config.optimizer,
        'metrics': (train_acc_fn, train_f1_fn, test_acc_fn, test_f1_fn, test_auc_pr_fn),
        'metric_names': ['train_acc', 'train_f1', 'test_acc', 'test_f1', 'test_auc_pr'],
        'loss_fn': criterion,
        'steps': config.fine_tune_steps,
        'lamb': config.lambda_reg,
        'lamb_entropy': config.lambda_entropy,
        'batch_size': config.batch_size,
        'lr': config.learning_rate * 0.5,  # Use a smaller learning rate for fine-tuning
        'weight_decay': config.weight_decay
    }
    
    fine_tune_results = model.fit(dataset, **fine_tune_options)
    
    # Evaluate fine-tuned model
    model.eval()
    with torch.no_grad():
        test_outputs = model(dataset['test_input'])
        test_preds = torch.argmax(test_outputs, dim=1)
        test_acc_finetuned = torch.mean((test_preds == dataset['test_label']).float()).item()
        test_f1_finetuned = f1_metric(model, dataset['test_input'], dataset['test_label']).item()
        test_auc_pr_finetuned = auc_pr_metric(model, dataset['test_input'], dataset['test_label']).item()
    
    print("\nFine-tuning complete. Evaluation results:")
    print(f"  Original model test accuracy: {test_acc:.4f}")
    print(f"  Pruned model test accuracy: {test_acc_pruned:.4f}")
    print(f"  Fine-tuned model test accuracy: {test_acc_finetuned:.4f}")
    print(f"  Original model test F1 score: {test_f1:.4f}")
    print(f"  Pruned model test F1 score: {test_f1_pruned:.4f}")
    print(f"  Fine-tuned model test F1 score: {test_f1_finetuned:.4f}")
    print(f"  Original model test AUC-PR: {test_auc_pr:.4f}")
    print(f"  Fine-tuned model test AUC-PR: {test_auc_pr_finetuned:.4f}")
    
    # Save the fine-tuned model
    torch.save(model.state_dict(), f'models/kan_label{target_label}_pruned_finetuned.pt')
    print(f"Fine-tuned model saved to 'models/kan_label{target_label}_pruned_finetuned.pt'")
    
    # Visualize the pruned and fine-tuned model
    model.eval()
    fig = model.plot(
        beta=50,
        scale=1,
        in_vars=feature_names[:min(10, config.feature_dim)],
        out_vars=output_names
    )
    plt.savefig('results/pruned_finetuned_model_structure.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Compare parameter counts
    original_params = sum(p.numel() for p in original_model.parameters() if p.requires_grad)
    pruned_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"\nModel size comparison:")
    print(f"  Original model parameters: {original_params}")
    print(f"  Pruned & fine-tuned model parameters: {pruned_params}")
    print(f"  Reduction: {100 * (1 - pruned_params / original_params):.2f}%")
else:
    print("\nModel pruning is disabled. Skipping pruning and fine-tuning.")

In [ ]:
# Cell 9: Symbolic Expression Extraction

if config.enable_symbolic:
    print("\nExtracting symbolic expressions from the model...")
    
    # Define the library of functions to use for symbolic regression
    lib = config.symbolic_library
    print(f"Using function library: {lib}")
    
    try:
        # Extract symbolic expressions from the model
        model.auto_symbolic(lib=lib)
        
        # Get the symbolic formulas
        formulas = model.symbolic_formula()[0]
        
        print("\nExtracted symbolic formulas:")
        for i, formula in enumerate(formulas):
            print(f"\nOutput {i} ({output_names[i]}):")
            print(formula)
            
            # Try to simplify the formula using sympy
            try:
                from sympy import simplify
                simplified = simplify(formula)
                print("\nSimplified:")
                print(simplified)
            except Exception as e:
                print(f"Could not simplify formula: {str(e)}")
        
        # Save the formulas to a text file
        with open(f'results/symbolic_formulas_label{target_label}.txt', 'w') as f:
            f.write(f"Symbolic formulas for Label {target_label} (1-vs-rest):\n")
            f.write("=" * 60 + "\n\n")
            
            for i, formula in enumerate(formulas):
                f.write(f"Output {i} ({output_names[i]}):\n")
                f.write(str(formula) + "\n\n")
                
                try:
                    from sympy import simplify
                    simplified = simplify(formula)
                    f.write("Simplified:\n")
                    f.write(str(simplified) + "\n\n")
                except:
                    f.write("Could not simplify formula.\n\n")
        
        print(f"\nFormulas saved to 'results/symbolic_formulas_label{target_label}.txt'")
        
        # Analyze the importance of input features
        print("\nAnalyzing feature importance based on symbolic expressions...")
        
        # This is a simple approach - count feature occurrences in formulas
        feature_counts = {}
        
        for i, formula in enumerate(formulas):
            formula_str = str(formula)
            
            for j, feature in enumerate(feature_names):
                if feature in formula_str:
                    if feature not in feature_counts:
                        feature_counts[feature] = 0
                    feature_counts[feature] += 1
        
        # Sort features by importance (occurrence count)
        sorted_features = sorted(feature_counts.items(), key=lambda x: x[1], reverse=True)
        
        print("\nTop 10 most important features:")
        for feature, count in sorted_features[:10]:
            print(f"  {feature}: {count} occurrences")
            
    except Exception as e:
        print(f"Error extracting symbolic expressions: {str(e)}")
        import traceback
        traceback.print_exc()
else:
    print("\nSymbolic expression extraction is disabled. Skipping.")

In [ ]:
# Cell 10: Create Training Visualization Video

try:
    import moviepy.video.io.ImageSequenceClip
    
    # Create a video from the saved training visualization images
    video_name = f'results/training_visualization_label{target_label}'
    fps = config.video_fps
    
    # Get all the image files from the folder
    files = os.listdir(config.img_folder)
    train_index = []
    
    for file in files:
        if file[0].isdigit() and file.endswith('.jpg'):
            train_index.append(int(file[:-4]))
    
    # Sort indices and create file paths
    train_index = np.sort(train_index)
    image_files = [f'{config.img_folder}/{idx}.jpg' for idx in train_index]
    
    if image_files:
        # Create the video clip and save it
        clip = moviepy.video.io.ImageSequenceClip.ImageSequenceClip(image_files, fps=fps)
        clip.write_videofile(f'{video_name}.mp4')
        
        print(f"\nTraining visualization video created: '{video_name}.mp4'")
        print(f"Video contains {len(image_files)} frames at {fps} fps")
    else:
        print("\nNo training visualization images found.")
except ImportError:
    print("\nMoviepy not available. Skipping video creation.")
except Exception as e:
    print(f"\nError creating training visualization video: {str(e)}")

In [ ]:
# Cell 11: Implement and Train a Neural Network for Comparison (Optional)

# Define a standard neural network with the same architecture as the KAN
class BrainVoxelNN(torch.nn.Module):
    def __init__(self, config):
        super(BrainVoxelNN, self).__init__()
        
        # Get network architecture from config
        layers = []
        
        for i in range(len(config.network_width) - 1):
            if i < len(config.network_width) - 2:
                # Hidden layers with ReLU activation
                layers.append(torch.nn.Linear(config.network_width[i], config.network_width[i+1]))
                layers.append(torch.nn.ReLU())
            else:
                # Output layer (no activation, will use softmax in loss function)
                layers.append(torch.nn.Linear(config.network_width[i], config.network_width[i+1]))
        
        self.model = torch.nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

def train_neural_network(model, dataset, config):
    """Train a standard neural network"""
    print("\nTraining standard neural network for comparison...")
    
    # Set up loss function with class weights
    class_weights = config.class_weights.to(device)
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
    
    # Create optimizer
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )
    
    # Get training data
    train_input = dataset['train_input']
    train_label = dataset['train_label']
    
    # Keep track of metrics
    results = {
        'train_acc': [],
        'train_f1': [],
        'test_acc': [],
        'test_f1': [],
        'test_auc_pr': [],
        'loss': []
    }
    
    # Training loop
    for step in range(config.train_steps):
        model.train()
        
        # Sample a batch
        if train_input.shape[0] <= config.batch_size:
            batch_input = train_input
            batch_label = train_label
        else:
            indices = torch.randperm(train_input.shape[0])[:config.batch_size]
            batch_input = train_input[indices]
            batch_label = train_label[indices]
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(batch_input)
        loss = criterion(outputs, batch_label)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        # Calculate metrics every few steps
        if (step + 1) % 5 == 0 or step == config.train_steps - 1:
            model.eval()
            with torch.no_grad():
                # Calculate training metrics
                train_outputs = model(train_input)
                train_preds = torch.argmax(train_outputs, dim=1)
                train_accuracy = torch.mean((train_preds == train_label).float()).item()
                
                # Calculate F1 score - manually to avoid using the KAN specific functions
                train_tp = torch.sum((train_preds == 1) & (train_label == 1)).float().item()
                train_fp = torch.sum((train_preds == 1) & (train_label == 0)).float().item()
                train_fn = torch.sum((train_preds == 0) & (train_label == 1)).float().item()
                
                train_precision = train_tp / (train_tp + train_fp) if (train_tp + train_fp) > 0 else 0
                train_recall = train_tp / (train_tp + train_fn) if (train_tp + train_fn) > 0 else 0
                train_f1 = 2 * train_precision * train_recall / (train_precision + train_recall) if (train_precision + train_recall) > 0 else 0
                
                # Calculate test metrics
                test_outputs = model(dataset['test_input'])
                test_preds = torch.argmax(test_outputs, dim=1)
                test_accuracy = torch.mean((test_preds == dataset['test_label']).float()).item()
                
                # Calculate test F1 score
                test_tp = torch.sum((test_preds == 1) & (dataset['test_label'] == 1)).float().item()
                test_fp = torch.sum((test_preds == 1) & (dataset['test_label'] == 0)).float().item()
                test_fn = torch.sum((test_preds == 0) & (dataset['test_label'] == 1)).float().item()
                
                test_precision = test_tp / (test_tp + test_fp) if (test_tp + test_fp) > 0 else 0
                test_recall = test_tp / (test_tp + test_fn) if (test_tp + test_fn) > 0 else 0
                test_f1 = 2 * test_precision * test_recall / (test_precision + test_recall) if (test_precision + test_recall) > 0 else 0
                
                # Calculate AUC-PR
                test_probs = torch.softmax(test_outputs, dim=1)[:, 1].cpu().numpy()
                test_labels = dataset['test_label'].cpu().numpy()
                test_auc_pr = average_precision_score(test_labels, test_probs)
            
            # Store metrics
            results['train_acc'].append(train_accuracy)
            results['train_f1'].append(train_f1)
            results['test_acc'].append(test_accuracy)
            results['test_f1'].append(test_f1)
            results['test_auc_pr'].append(test_auc_pr)
            results['loss'].append(loss.item())
            
            # Print progress
            print(f"Step {step+1}/{config.train_steps}, "
                  f"Loss: {loss.item():.4f}, "
                  f"Train Acc: {train_accuracy:.4f}, "
                  f"Test Acc: {test_accuracy:.4f}, "
                  f"Test F1: {test_f1:.4f}, "
                  f"Test AUC-PR: {test_auc_pr:.4f}")
    
    # Final evaluation
    model.eval()
    with torch.no_grad():
        test_outputs = model(dataset['test_input'])
        test_preds = torch.argmax(test_outputs, dim=1)
        test_accuracy = torch.mean((test_preds == dataset['test_label']).float()).item()
        
        # Calculate test F1 score
        test_tp = torch.sum((test_preds == 1) & (dataset['test_label'] == 1)).float().item()
        test_fp = torch.sum((test_preds == 1) & (dataset['test_label'] == 0)).float().item()
        test_fn = torch.sum((test_preds == 0) & (dataset['test_label'] == 1)).float().item()
        
        test_precision = test_tp / (test_tp + test_fp) if (test_tp + test_fp) > 0 else 0
        test_recall = test_tp / (test_tp + test_fn) if (test_tp + test_fn) > 0 else 0
        test_f1 = 2 * test_precision * test_recall / (test_precision + test_recall) if (test_precision + test_recall) > 0 else 0
        
        # Calculate AUC-PR
        test_probs = torch.softmax(test_outputs, dim=1)[:, 1].cpu().numpy()
        test_labels = dataset['test_label'].cpu().numpy()
        test_auc_pr = average_precision_score(test_labels, test_probs)
    
    print("\nNeural Network training completed.")
    print(f"Final test accuracy: {test_accuracy:.4f}")
    print(f"Final test F1 score: {test_f1:.4f}")
    print(f"Final test AUC-PR: {test_auc_pr:.4f}")
    
    # Save the model
    torch.save(model.state_dict(), f'models/nn_label{target_label}_final.pt')
    
    return model, results

# Create and train the neural network
nn_model = BrainVoxelNN(config).to(device)
nn_model, nn_results = train_neural_network(nn_model, dataset, config)

# Compare KAN and NN results
def compare_models(kan_results, nn_results):
    """Compare KAN and Neural Network results"""
    plt.figure(figsize=(15, 12))
    
    # Compare test accuracy
    plt.subplot(2, 2, 1)
    plt.plot(kan_results['test_acc'], label='KAN')
    plt.plot(nn_results['test_acc'], label='Neural Network')
    plt.title('Test Accuracy Comparison')
    plt.xlabel('Steps (x5)')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    # Compare test F1 score
    plt.subplot(2, 2, 2)
    plt.plot(kan_results['test_f1'], label='KAN')
    plt.plot(nn_results['test_f1'], label='Neural Network')
    plt.title('Test F1 Score Comparison')
    plt.xlabel('Steps (x5)')
    plt.ylabel('F1 Score')
    plt.legend()
    plt.grid(True)
    
    # Compare test AUC-PR
    plt.subplot(2, 2, 3)
    plt.plot(kan_results['test_auc_pr'], label='KAN')
    plt.plot(nn_results['test_auc_pr'], label='Neural Network')
    plt.title('Test AUC-PR Comparison')
    plt.xlabel('Steps (x5)')
    plt.ylabel('AUC-PR')
    plt.legend()
    plt.grid(True)
    
    # Compare loss
    plt.subplot(2, 2, 4)
    plt.plot(kan_results['loss'], label='KAN')
    plt.plot(nn_results['loss'], label='Neural Network')
    plt.title('Training Loss Comparison')
    plt.xlabel('Steps (x5)')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(f'results/kan_vs_nn_comparison_label{target_label}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print final metrics comparison
    print("\nModel Comparison Summary:")
    print(f"{'Metric':<15} {'KAN':<10} {'Neural Network':<15} {'Difference':<10}")
    print("-" * 50)
    
    # Function to get the last value of a metric
    def get_last(results, metric):
        return results[metric][-1] if results[metric] else 0
    
    metrics = ['test_acc', 'test_f1', 'test_auc_pr']
    metric_names = ['Test Accuracy', 'Test F1', 'Test AUC-PR']
    
    for name, metric in zip(metric_names, metrics):
        kan_value = get_last(kan_results, metric)
        nn_value = get_last(nn_results, metric)
        diff = kan_value - nn_value
        diff_str = f"{diff:.4f} ({'better' if diff > 0 else 'worse'})"
        print(f"{name:<15} {kan_value:.4f}     {nn_value:.4f}        {diff_str}")

# Compare the models
compare_models(results, nn_results)

In [ ]:
# Cell 12: Final Summary and Conclusion

print("\n" + "="*50)
print(f"Final Summary for Label {target_label} (1-vs-rest Classification)")
print("="*50)

# Summarize dataset information
print("\nDataset Information:")
print(f"  Label ID: {target_label}")
train_pos = torch.sum(dataset['train_label'] == 1).item()
train_neg = torch.sum(dataset['train_label'] == 0).item()
test_pos = torch.sum(dataset['test_label'] == 1).item()
test_neg = torch.sum(dataset['test_label'] == 0).item()

print(f"  Training samples: {len(dataset['train_label'])} ({train_pos} positive, {train_neg} negative)")
print(f"  Test samples: {len(dataset['test_label'])} ({test_pos} positive, {test_neg} negative)")
print(f"  Input features: {config.feature_dim}")
print(f"  Class ratio: 1:{config.negative_ratio} (positive:negative)")

# Summarize KAN model performance
print("\nKAN Model Performance:")
final_test_acc = results['test_acc'][-1] if results['test_acc'] else 0
final_test_f1 = results['test_f1'][-1] if results['test_f1'] else 0
final_test_auc_pr = results['test_auc_pr'][-1] if results['test_auc_pr'] else 0

print(f"  Test Accuracy: {final_test_acc:.4f}")
print(f"  Test F1 Score: {final_test_f1:.4f}")
print(f"  Test AUC-PR: {final_test_auc_pr:.4f}")

if config.enable_pruning:
    # Get results from pruned & fine-tuned model
    pruned_test_acc = test_acc_finetuned if 'test_acc_finetuned' in locals() else 0
    pruned_test_f1 = test_f1_finetuned if 'test_f1_finetuned' in locals() else 0
    pruned_test_auc_pr = test_auc_pr_finetuned if 'test_auc_pr_finetuned' in locals() else 0
    
    print("\nPruned & Fine-tuned KAN Model:")
    print(f"  Test Accuracy: {pruned_test_acc:.4f}")
    print(f"  Test F1 Score: {pruned_test_f1:.4f}")
    print(f"  Test AUC-PR: {pruned_test_auc_pr:.4f}")
    
    # Parameter reduction
    if 'original_params' in locals() and 'pruned_params' in locals():
        print(f"  Parameter reduction: {100 * (1 - pruned_params / original_params):.2f}%")
        print(f"  Original parameters: {original_params}")
        print(f"  Pruned parameters: {pruned_params}")

# Summarize neural network performance (if available)
if 'nn_results' in locals():
    nn_final_test_acc = nn_results['test_acc'][-1] if nn_results['test_acc'] else 0
    nn_final_test_f1 = nn_results['test_f1'][-1] if nn_results['test_f1'] else 0
    nn_final_test_auc_pr = nn_results['test_auc_pr'][-1] if nn_results['test_auc_pr'] else 0
    
    print("\nNeural Network Model Performance:")
    print(f"  Test Accuracy: {nn_final_test_acc:.4f}")
    print(f"  Test F1 Score: {nn_final_test_f1:.4f}")
    print(f"  Test AUC-PR: {nn_final_test_auc_pr:.4f}")
    
    # Compare KAN vs NN
    print("\nKAN vs Neural Network:")
    print(f"  Accuracy: {final_test_acc - nn_final_test_acc:.4f} difference ({'better' if final_test_acc > nn_final_test_acc else 'worse'})")
    print(f"  F1 Score: {final_test_f1 - nn_final_test_f1:.4f} difference ({'better' if final_test_f1 > nn_final_test_f1 else 'worse'})")
    print(f"  AUC-PR: {final_test_auc_pr - nn_final_test_auc_pr:.4f} difference ({'better' if final_test_auc_pr > nn_final_test_auc_pr else 'worse'})")

# Print optimal threshold
if 'best_threshold' in locals():
    print(f"\nOptimal classification threshold (based on F1): {best_threshold:.4f}")

# Summarize PR curve analysis
if 'auc_pr' in locals():
    print(f"\nPrecision-Recall Analysis:")
    print(f"  AUC-PR: {auc_pr:.4f}")
    if 'best_f1' in locals() and 'best_threshold' in locals():
        print(f"  Best F1 Score: {best_f1:.4f} (at threshold = {best_threshold:.4f})")

# Print symbolic expression summary (if available)
if config.enable_symbolic and 'formulas' in locals():
    print("\nSymbolic Expressions:")
    print(f"  Extracted formulas saved to 'results/symbolic_formulas_label{target_label}.txt'")
    
    if 'sorted_features' in locals():
        print("\n  Top 5 most important features:")
        for i, (feature, count) in enumerate(sorted_features[:5]):
            print(f"    {i+1}. {feature}: {count} occurrences")

# Print file locations
print("\nOutput Files:")
print(f"  Models saved to 'models/' directory")
print(f"  Results and visualizations saved to 'results/' directory")
if config.save_figures:
    print(f"  Training visualization saved as 'results/training_visualization_label{target_label}.mp4'")

print("\nConclusion:")
print("  The KAN approach demonstrates the ability to create interpretable models")
print("  for brain voxel classification, with the added benefit of being able to")
print("  extract symbolic formulas and prune unnecessary connections.")
print("  This implementation shows how to apply KAN to 1-vs-rest classification")
print("  for complex high-dimensional neuroimaging data.")
print("="*50)

# Create a timestamp for this run
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"\nExecution completed at: {timestamp}")